# 🧠 TARS HELIX LITE — Тренировка на Colab

**296M параметров | Dual-SSM (SSD + WKV) | BitNet Ternary | SwiGLU**

| Что | Где |
|---|---|
| Код | `/content/TarsSSM-Py/` |
| Чекпоинты | `MyDrive/TarsData/models/tars_lite/` |
| Логи | `train_lite.log` |

### Уровни
| Уровень | Время | GPU |
|---|---|---|
| `small` | ~15 мин | Любой |
| `medium` | ~3 часа | T4+ |
| `max` | ~15 часов | L4/A100 |
| `marathon` | ~96 часов | A100 |

### Инструкция
1. **Runtime → Change runtime type → T4 GPU** (или L4/A100)
2. **Выберите уровень** в ячейке ниже и запустите
3. Если отключилось — просто **перезапустите ячейку** (продолжит с чекпоинта)

In [ ]:
#@title 🚀 **HELIX LITE — Тренировка** { display-mode: "form" }

#@markdown ### Настройки
LEVEL = "medium" #@param ["small", "medium", "max", "marathon"] {type:"string"}
#@markdown - `small` — smoke test (~15 мин)
#@markdown - `medium` — стандарт (~3 часа)
#@markdown - `max` — продакшн (~15 часов)
#@markdown - `marathon` — 4 дня (~96ч)

import os, sys, time, shutil, subprocess
from pathlib import Path

# ============================================================
# 1. GOOGLE DRIVE
# ============================================================
print('=' * 60)
print('  🧠 TARS HELIX LITE — Тренировка')
print('=' * 60)
print()

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = Path('/content/drive/MyDrive/TarsData')
DRIVE_MODELS = DRIVE_ROOT / 'models' / 'tars_lite'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)

print(f'  ☁️  Drive: {DRIVE_ROOT}')
existing_ckpts = list(DRIVE_MODELS.glob('*.pt'))
if existing_ckpts:
    mb = sum(f.stat().st_size for f in existing_ckpts) / 1024**2
    print(f'  💾 Чекпоинты: {len(existing_ckpts)} файлов, {mb:.0f} MB')
print()

# ============================================================
# 2. GPU CHECK
# ============================================================
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'No GPU'
print()

# ============================================================
# 3. CLONE / UPDATE CODE
# ============================================================
os.chdir('/content')
REPO = 'TarsSSM-Py'
if os.path.exists(REPO):
    print('🔄 Обновление кода...')
    os.chdir(f'/content/{REPO}')
    !git stash 2>/dev/null; git pull --rebase origin main 2>&1 | tail -3; git stash pop 2>/dev/null || true
else:
    print('📥 Клонирование...')
    !git clone https://github.com/Vazilll/TarsSSM-Py.git
    os.chdir(f'/content/{REPO}')

ROOT = Path(f'/content/{REPO}')
print(f'  📁 Root: {ROOT}')
print()

# ============================================================
# 4. SYMLINK: models/ → Drive
# ============================================================
def safe_symlink(local, drive_target):
    """Symlink local → Drive (move existing files first)"""
    if local.is_symlink():
        return
    if local.exists():
        for f in local.rglob('*'):
            if f.is_file():
                rel = f.relative_to(local)
                dest = drive_target / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.move(str(f), str(dest))
        shutil.rmtree(str(local))
    local.symlink_to(drive_target)

# models/tars_lite/ → Drive checkpoints
(DRIVE_ROOT / 'models' / 'tars_lite').mkdir(parents=True, exist_ok=True)
safe_symlink(ROOT / 'models', DRIVE_ROOT / 'models')
print('  🔗 models/ → Drive/TarsData/models/')
print()

# ============================================================
# 5. INSTALL DEPS
# ============================================================
print('  📦 Установка зависимостей...')
!pip install -q einops 2>/dev/null
print('  ✅ Готово')
print()

# ============================================================
# 6. KEEP-ALIVE (анти-таймаут)
# ============================================================
import threading
def _keep_alive():
    while True:
        time.sleep(600)
        elapsed = time.time() - _start
        h, m = int(elapsed // 3600), int((elapsed % 3600) // 60)
        print(f'  ⏰ Keep-alive: {h}h {m}m — обучение продолжается...', flush=True)

_start = time.time()
_ka = threading.Thread(target=_keep_alive, daemon=True)
_ka.start()

# ============================================================
# 7. ОБУЧЕНИЕ через local_train.py
# ============================================================
print('=' * 60)
print(f'  🎓 HELIX LITE: уровень={LEVEL}')
print(f'  ☁️  Чекпоинты сохраняются на Drive автоматически')
print('=' * 60)
print(flush=True)

cmd = [
    sys.executable, '-u', str(ROOT / 'local_train.py'),
    '--level', LEVEL,
    '--drive', 'colab',
    '--resume',
    '--no-git-pull',  # already pulled above
]

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

process = subprocess.Popen(
    cmd, cwd=str(ROOT),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env=env, bufsize=1, universal_newlines=True,
)

for line in process.stdout:
    print(line, end='', flush=True)

returncode = process.wait()

# ============================================================
# 8. РЕЗУЛЬТАТ
# ============================================================
elapsed = time.time() - _start
hours = elapsed / 3600
minutes = elapsed / 60

print()
print('=' * 60)
if returncode == 0:
    print(f'  ✅ ОБУЧЕНИЕ ЗАВЕРШЕНО за {hours:.1f}ч ({minutes:.0f} мин)!')
    print()
    for f in sorted(DRIVE_MODELS.rglob('*.pt')):
        mb = f.stat().st_size / 1024**2
        print(f'    💾 {f.name}: {mb:.1f} MB')
    print()
    print('  📁 Чекпоинты: MyDrive/TarsData/models/tars_lite/')
else:
    print(f'  ⚠️  Ошибка (код {returncode}) — {minutes:.0f} мин')
    print('  Чекпоинты сохранены на Drive.')
    print('  ⏯️  Перезапустите ячейку — продолжит с чекпоинта.')
print('=' * 60)

In [ ]:
#@title 📊 Статус обучения { display-mode: "form" }
from pathlib import Path
import json

ROOT = Path('/content/TarsSSM-Py')
DRIVE_MODELS = Path('/content/drive/MyDrive/TarsData/models/tars_lite')

print('=' * 50)
print('  📊 Статус обучения HELIX LITE')
print('=' * 50)

# Train state
state_file = ROOT / 'train_state.json'
if state_file.exists():
    state = json.loads(state_file.read_text())
    print(f'  🏁 Epoch:    {state.get("current_epoch", 0)}')
    print(f'  📉 Best loss: {state.get("best_loss", "?")}')
    print(f'  🔢 Steps:    {state.get("total_steps", 0)}')
    print()

# Checkpoints
if DRIVE_MODELS.exists():
    for f in sorted(DRIVE_MODELS.rglob('*.pt')):
        mb = f.stat().st_size / 1024**2
        print(f'  💾 {f.name}: {mb:.1f} MB')
else:
    print('  (нет чекпоинтов)')

# GPU
try:
    import torch
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'\n  🎮 VRAM: {alloc:.1f}/{total:.1f} GB')
except: pass

# Log tail
print()
print('  📋 Последние строки лога:')
log = ROOT / 'train_lite.log'
if log.exists():
    lines = log.read_text(encoding='utf-8', errors='replace').strip().split('\n')
    for l in lines[-15:]:
        print(f'    {l}')
else:
    print('    (нет лога)')
print('=' * 50)

In [ ]:
#@title 📥 Скачать чекпоинты { display-mode: "form" }
from google.colab import files
from pathlib import Path
import zipfile, os

DRIVE_MODELS = Path('/content/drive/MyDrive/TarsData/models/tars_lite')

if DRIVE_MODELS.exists() and list(DRIVE_MODELS.glob('*.pt')):
    zip_path = '/content/tars_helix_lite.zip'
    print('📦 Упаковка...')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in DRIVE_MODELS.rglob('*'):
            if f.is_file():
                zf.write(f, f.relative_to(DRIVE_MODELS))
                print(f'  + {f.name} ({f.stat().st_size / 1024**2:.1f} MB)')
    
    total_mb = os.path.getsize(zip_path) / 1024**2
    print(f'\n💾 Итого: {total_mb:.0f} MB')
    print('📥 Скачивание...')
    files.download(zip_path)
else:
    print('⚠️  Чекпоинты не найдены. Сначала запустите обучение.')

In [ ]:
#@title 🔬 Smoke Test (без тренировки) { display-mode: "form" }
import os, sys
os.chdir('/content/TarsSSM-Py')
sys.path.insert(0, '/content/TarsSSM-Py')

!python -u local_train.py --test-only

In [ ]:
#@title 📊 Параметры модели { display-mode: "form" }
import os, sys
os.chdir('/content/TarsSSM-Py')
sys.path.insert(0, '/content/TarsSSM-Py')

!python -u local_train.py --count-params

### 📝 Полезные команды
```python
# Smoke test
!cd /content/TarsSSM-Py && python -u local_train.py --test-only

# Параметры
!cd /content/TarsSSM-Py && python -u local_train.py --count-params

# Быстрый тест
!cd /content/TarsSSM-Py && python -u local_train.py --level small --drive colab --resume

# Стандарт
!cd /content/TarsSSM-Py && python -u local_train.py --level medium --drive colab --resume

# Максимум
!cd /content/TarsSSM-Py && python -u local_train.py --level max --drive colab --resume

# Марафон (4 дня)
!cd /content/TarsSSM-Py && python -u local_train.py --level marathon --drive colab --resume

# Лог
!tail -30 /content/TarsSSM-Py/train_lite.log
```